# NLP Project 2 – Whisper ASR Evaluation on Swiss German

**Dataset:** STT4SG-350 (Swiss German, 7 Dialektregionen)  
**Forschungsfrage:** Wie verändert sich die ASR-Qualität von Whisper v1→v2→v3 auf Schweizerdeutsch, und welche Fehlertypen dominieren?  
**Modelle:** `openai/whisper-large-v1`, `openai/whisper-large-v2`, `openai/whisper-large-v3`  
**Metrik:** WER (Word Error Rate), optional CER  

**Authors:** Natalie Jakab  (jakabnat) & Sarruja Sabesan (sabessar)
**Kurs:** Introduction to NLP – FS 2026, ZHAW

---

## Notebook-Struktur

| Abschnitt | Was passiert |
|---|---|
| Setup | Imports, Pfade, Hilfsfunktionen |
| Test-Run (lokal) | 10 Samples, 1 Modell → schnell validieren |
| Full-Run (Colab) | Alle Samples, alle 3 Modelle, Checkpoint alle 200 |
| Ergebnisse & WER | CSV einlesen, WER overall + per Dialektregion |
| Visualisierung | Grouped Bar Chart WER pro Modell & Region |
| CER (optional) | Character Error Rate falls Zeit bleibt |

---
## 0. Setup

In [1]:
# ── Installations (nur beim ersten Mal / auf Colab nötig) ──────────────────
!pip install transformers==4.36.0 --quiet
!pip install torch datasets jiwer soundfile librosa ffmpeg-python --quiet
!apt-get install ffmpeg -y -q

# ffmpeg (externes Tool) muss separat installiert werden (in PowerShell/Anaconda Prompt):
# conda install -c conda-forge ffmpeg -y

# evtl. Kernel neustarten wenns import nicht direkt funktionert



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 2.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 59.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:0000:0100:01
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.


In [2]:
import os
import csv
import time
import torch
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import transformers
from transformers import pipeline #für Whisper
from jiwer import wer, cer

import os
import warnings

warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print(f'Tranformer version: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'Using device:    {"GPU" if DEVICE == 0 else "CPU"}')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-04-02 07:39:42.015288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775115582.246531      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775115582.309872      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775115582.812180      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00

PyTorch version: 2.10.0+cu128
CUDA available:  True
Tranformer version: 4.36.0
GPU:             Tesla T4
Using device:    GPU


In [3]:
# Kaggle Pfade
DATA_DIR    = '/kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/'
AUDIO_DIR   = '/kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/clips__test/'
RESULTS_DIR = '/kaggle/working/'
TEST_FILE   = '/kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/test.tsv'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Data dir:    {DATA_DIR}')
print(f'Audio dir:   {AUDIO_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Test file:   {TEST_FILE}')

Data dir:    /kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/
Audio dir:   /kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/clips__test/
Results dir: /kaggle/working/
Test file:   /kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/test.tsv


In [4]:
import os
for root, dirs, files in os.walk('/kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/'):
    for f in files[:3]:
        print(os.path.join(root, f))
    break

/kaggle/input/datasets/sarruja/data-nlp-project2/data_kaggle_p2/test.tsv


In [5]:
df_test = pd.read_csv(TEST_FILE, sep='\t')
print(f'Test set: {len(df_test)} Samples')
print(f'Columns:  {df_test.columns.tolist()}')
df_test.head(3)

Test set: 24605 Samples
Columns:  ['path', 'duration', 'sentence', 'sentence_source', 'client_id', 'dialect_region', 'canton', 'zipcode', 'age', 'gender']


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender
0,8800b4de-fa22-4fdc-9f29-457c4010fd57/d57790928...,4.478667,"Die Steuerfrage wäre eine andere, die Planung ...",parliament,8800b4de-fa22-4fdc-9f29-457c4010fd57,Basel,BS,4001,fourties,female
1,887b50f8-215b-4a1d-8f32-13516da6506f/73ca6c9b0...,6.134667,"Ich bin der Meinung, dass ihr noch nicht alle ...",parliament,887b50f8-215b-4a1d-8f32-13516da6506f,Bern,BE,3270,sixties,female
2,31cab952-98eb-45cb-a243-c951519c5c40/3aca44313...,5.126667,Dann änderte sie ihre Meinung und verlangte di...,news_switz,31cab952-98eb-45cb-a243-c951519c5c40,Innerschweiz,LU,6004,twenties,female


In [6]:
# ── Spalten prüfen & anpassen ──────────────────────────────────────────────
# STT4SG-350 test.csv hat typischerweise diese Spalten:
#   path, sentence, locale (Dialektregion)
# Falls die Spaltennamen anders sind → hier anpassen:
PATH_COL   = 'path'       # Dateiname des Audioclips (z.B. "abc123.mp3")
REF_COL    = 'sentence'   # Ground-Truth Transkription (Hochdeutsch)
REGION_COL = 'canton'     # Dialektregion (z.B. "BE", "ZH", ...)

print('Dialektregionen:', df_test[REGION_COL].unique())
print('Verteilung:')
print(df_test[REGION_COL].value_counts())

Dialektregionen: ['BS' 'BE' 'LU' 'SO' 'SG' 'ZG' 'VS' 'AG' 'TG' 'BL' 'ZH' 'GR' 'SH' 'UR'
 'TI' 'GL']
Verteilung:
canton
VS    3515
GR    3159
ZH    2532
BE    2494
SG    2450
BS    2261
LU    1704
AG    1550
BL    1254
SO     813
ZG     723
TG     721
UR     365
GL     364
TI     356
SH     344
Name: count, dtype: int64


---
## 1. Test-Run (lokal) – 10 Samples, 1 Modell

Zuerst lokal mit wenigen Samples validieren, dass die Pipeline funktioniert.  
Wenn das klappt → auf Colab mit dem Full-Run.

In [7]:
# ── Hilfsfunktion: Whisper-Pipeline laden ─────────────────────────────────
def load_whisper(model_id):
    """Lädt Whisper als HuggingFace pipeline."""
    print(f'Lade Modell: {model_id} ...')
    asr = pipeline(
        task='automatic-speech-recognition',
        model=model_id,
        device=DEVICE,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    # Sprache auf Deutsch zwingen (wichtig für Schweizerdeutsch!)
    asr.model.config.forced_decoder_ids = (
        asr.tokenizer.get_decoder_prompt_ids(language='german', task='transcribe')
    )
    print(f'Modell geladen: {model_id}')
    return asr


# ── Hilfsfunktion: Audio-Pfad auflösen ────────────────────────────────────
def resolve_audio_path(row):
    """Gibt den vollen Pfad zum Audioclip zurück."""
    return os.path.join(AUDIO_DIR, row[PATH_COL])


# ── Hilfsfunktion: Einen Clip transkribieren ───────────────────────────────
def transcribe_clip(asr_pipeline, audio_path):
    """Transkribiert einen Audioclip und gibt den Text zurück."""
    result = asr_pipeline(audio_path)
    return result['text'].strip()


print('Hilfsfunktionen definiert.')

Hilfsfunktionen definiert.


In [8]:
# ── Test-Run: 10 Samples mit whisper-large-v2 ─────────────────────────────
print("Can be skipped - only for tesitng")

N_TEST = 10
TEST_MODEL = 'openai/whisper-large-v3'

df_sample = df_test.head(N_TEST).copy()

asr = load_whisper(TEST_MODEL)

predictions = []
for i, row in df_sample.iterrows():
    audio_path = resolve_audio_path(row)
    if not os.path.exists(audio_path):
        print(f'  ⚠️  Datei nicht gefunden: {audio_path}')
        predictions.append('')
        continue
    hyp = transcribe_clip(asr, audio_path)
    predictions.append(hyp)
    print(f'  [{i+1}/{N_TEST}] REF: {row[REF_COL]}')
    print(f'         HYP: {hyp}')
    print()

df_sample['hypothesis'] = predictions

# WER berechnen
refs = df_sample[REF_COL].str.lower().tolist()
hyps = df_sample['hypothesis'].str.lower().tolist()
test_wer = wer(refs, hyps)
print(f'Test-Run WER ({N_TEST} Samples, {TEST_MODEL}): {test_wer:.4f}')


Can be skipped - only for tesitng
Lade Modell: openai/whisper-large-v3 ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Modell geladen: openai/whisper-large-v3
  [1/10] REF: Die Steuerfrage wäre eine andere, die Planung eine nachhaltigere.
         HYP: Die Steuerfrage wären andere, die Planungen nachhaltiger.

  [2/10] REF: Ich bin der Meinung, dass ihr noch nicht alle Sparmöglichkeiten ausgeschöpft habt.
         HYP: Ich bin der Meinung, dass ihr noch nicht alle Sparmöglichkeiten ausgeschöpft habt.

  [3/10] REF: Dann änderte sie ihre Meinung und verlangte die Tausender zurück.
         HYP: Dann hat sie ihre Meinung geändert und den Tausender zurückverlangt.

  [4/10] REF: Und neue Türen haben sich nicht geöffnet.
         HYP: Und neue Türen haben sich nicht geöffnet.

  [5/10] REF: Der Gemeinderat nimmt es auch als Postulat entgegen.
         HYP: Der Gemeindetrode nimmt Sauer das Postulat entgegen.

  [6/10] REF: Der Syrer soll Teil dieser Gruppe gewesen sein.
         HYP: Der Sauer soll ein Teil dieser Gruppe gewesen sein.

  [7/10] REF: Land ist ein knappes Gut und nicht vermehrbar.
         H

---
## 2. Full-Run (Colab) – Alle Samples, alle 3 Modelle

⚠️ **Dieser Block ist für Colab mit GPU (T4).**  
Laufzeit ca. 3–6h total (alle 3 Modelle).  
Checkpoints werden alle 200 Samples gespeichert → bei Absturz kann man weitermachen.

**Wie starten:**
1. Google Drive mounten (Zeilen im Setup oben einkommentieren)
2. Alle Zellen von oben nach unten ausführen
3. Dann diese Zelle starten

In [8]:
# ── Konfig Full-Run ────────────────────────────────────────────────────────
MODELS = [
   # 'openai/whisper-large',  # is v1
   # 'openai/whisper-large-v2',
    'openai/whisper-large-v3',
]
CHECKPOINT_EVERY = 20  # alle 20 Samples speichern


def get_checkpoint_path(model_id):
    """Gibt den Checkpoint-Pfad für ein Modell zurück."""
    safe_name = model_id.replace('/', '_')
    return os.path.join(RESULTS_DIR, f'checkpoint_{safe_name}.csv')


def load_checkpoint(model_id):
    """Lädt bereits fertige Samples aus dem Checkpoint (falls vorhanden)."""
    path = get_checkpoint_path(model_id)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'  Checkpoint gefunden: {len(df)} Samples bereits verarbeitet.')
        return df
    return pd.DataFrame()


def save_checkpoint(rows, model_id):
    """Speichert aktuelle Ergebnisse als Checkpoint-CSV."""
    path = get_checkpoint_path(model_id)
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    print(f'Checkpoint gespeichert: {path} ({len(df)} Zeilen)', flush=True)

print('Full-Run Konfig bereit.')
print(f'Modelle: {MODELS}')
print(f'Checkpoint alle {CHECKPOINT_EVERY} Samples → {RESULTS_DIR}')

Full-Run Konfig bereit.
Modelle: ['openai/whisper-large-v3']
Checkpoint alle 20 Samples → /kaggle/working/


### Backupfile lesen und in Working Direcotry kopieren

In [9]:
backup_path = "/kaggle/input/datasets/sarruja/v3-backup/checkpoint_openai_whisper-large-v3_backup.csv"

df_done = pd.read_csv(backup_path)
print(len(df_done))

16680


In [10]:
import shutil

shutil.copy(
    backup_path,
    "/kaggle/working/checkpoint_openai_whisper-large-v3.csv"
)


print("Checkpoint wiederhergestellt!")

Checkpoint wiederhergestellt!


In [11]:
# ── Full-Run Loop ──────────────────────────────────────────────────────────
for model_id in MODELS:
    print(f'\n{"="*60}')
    print(f'Modell: {model_id}')
    print(f'{"="*60}')

    # Checkpoint laden (falls schon Fortschritt vorhanden)
    df_done = load_checkpoint(model_id)
    done_paths = set(df_done[PATH_COL].tolist()) if len(df_done) > 0 else set()

    # Noch ausstehende Samples
    df_todo = df_test[~df_test[PATH_COL].isin(done_paths)].reset_index(drop=True)
    print(f'Noch zu verarbeiten: {len(df_todo)} / {len(df_test)} Samples')

    if len(df_todo) == 0:
        print('  → Bereits fertig, überspringe.')
        continue

    # Modell laden
    asr = load_whisper(model_id)

    rows = df_done.to_dict('records')  # bisherige Ergebnisse
    t_start = time.time()

    for i, row in df_todo.iterrows():
        audio_path = resolve_audio_path(row)

        if not os.path.exists(audio_path):
            hyp = ''
        else:
            try:
                hyp = transcribe_clip(asr, audio_path)
            except Exception as e:
                print(f'  ⚠️  Fehler bei {audio_path}: {e}')
                hyp = ''

        rows.append({
            PATH_COL:     row[PATH_COL],
            REF_COL:      row[REF_COL],
            REGION_COL:   row[REGION_COL],
            'model':      model_id,
            'hypothesis': hyp,
        })

        # Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(rows, model_id)
            elapsed = time.time() - t_start
            print(f'  Checkpoint: {i+1}/{len(df_todo)} Samples | {elapsed/60:.1f} min')

    # Finales Speichern
    save_checkpoint(rows, model_id)
    print(f'  ✅ Fertig: {len(rows)} Samples gespeichert → {get_checkpoint_path(model_id)}')

    # Modell aus GPU-Speicher entfernen
    del asr
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('\n✅ Full-Run abgeschlossen!')


Modell: openai/whisper-large-v3
  Checkpoint gefunden: 16680 Samples bereits verarbeitet.
Noch zu verarbeiten: 7925 / 24605 Samples
Lade Modell: openai/whisper-large-v3 ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Modell geladen: openai/whisper-large-v3
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16700 Zeilen)
  Checkpoint: 20/7925 Samples | 0.4 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16720 Zeilen)
  Checkpoint: 40/7925 Samples | 0.7 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16740 Zeilen)
  Checkpoint: 60/7925 Samples | 1.0 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16760 Zeilen)
  Checkpoint: 80/7925 Samples | 1.2 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16780 Zeilen)
  Checkpoint: 100/7925 Samples | 1.5 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16800 Zeilen)
  Checkpoint: 120/7925 Samples | 1.8 min
Checkpoint gespeichert: /kaggle/working/checkpoint_openai_whisper-large-v3.csv (16820 Zeilen)
  Checkpoint: 140/7925 Samples | 2.1 min
Checkpoint gespeich